# Урок 12. Кратчайшие пути

11 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 11](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-11.ipynb) · [Урок 13 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-13.ipynb)

---

Задача о кратчайшем пути во взвешенном графе. Алгоритм Дейкстры на понятном уровне. Классические задачи про дороги и стоимость маршрута.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-12", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Кратчайший — не значит «с наименьшим числом дорог»

В прошлом уроке обход в ширину находил путь с наименьшим количеством
рёбер. Но если у дорог разная длина, это уже не помогает: две короткие
дороги могут оказаться выгоднее одной длинной.

Такой граф называется **взвешенным**: у каждого ребра есть число —
длина, время, стоимость. Задача о кратчайшем пути — найти маршрут
с наименьшей суммой весов.

```
       4
  А ───────── Б
  │           │
 1│           │2
  │     2     │
  В ───────── Г

  А → Б напрямую: 4
  А → В → Г → Б:  1 + 2 + 2 = 5
```

Здесь короче прямой путь. Поменяйте вес А–Б на 6 — и выгоднее станет
объезд из трёх дорог.

### Как хранить веса

**Весовая матрица**: на пересечении i и j — вес ребра, ноль или
прочерк, если дороги нет.

```
     А   Б   В   Г
  А  0   4   1   0
  Б  4   0   0   2
  В  1   0   0   2
  Г  0   2   2   0
```

**Список смежности с весами**: словарь «вершина → словарь
соседей с весами».

```python
{"А": {"Б": 4, "В": 1}, "Б": {"А": 4, "Г": 2}, ...}
```

В задачах ЕГЭ граф обычно дают именно таблицей, а прочерк означает
«дороги нет» — не «дорога длиной ноль». Это первая ловушка.

### Алгоритм Дейкстры

Эдсгер Дейкстра придумал его в 1956 году, обдумывая, как показать
возможности новой машины, — по его собственным словам, за двадцать
минут в кафе. Идея простая до наглости:

1. всем вершинам ставим метку «расстояние от старта»: старту — 0,
   остальным — бесконечность;
2. выбираем **непосещённую вершину с наименьшей меткой**;
3. для каждого её соседа проверяем: не короче ли добраться туда
   через неё? Если короче — уменьшаем метку соседа;
4. отмечаем вершину посещённой и возвращаемся к шагу 2.

Когда вершина выбрана на шаге 2, её метка окончательна: более
короткого пути уже не найдётся. Это и есть суть алгоритма.

### Почему это работает

Все веса неотрицательны, значит, любой путь только удлиняется.
Если до вершины X мы нашли расстояние 7 и это наименьшая метка среди
непосещённых, то любой другой путь до X пойдёт через вершину с меткой
не меньше 7 — и будет не короче.

Отсюда важное ограничение: **при отрицательных весах Дейкстра
не работает**. Для таких графов есть другие алгоритмы, помедленнее.

### Ручной приём для экзамена

На бумаге удобно вести таблицу: строки — шаги, столбцы — вершины.
На каждом шаге выбираем минимум среди непосещённых, обновляем соседей,
вычёркиваем столбец.

```
шаг   А    Б    В    Г     выбрали
 0    0    ∞    ∞    ∞        А
 1    –    4    1    ∞        В
 2    –    4    –    3        Г
 3    –    4    –    –        Б (через Г тоже 5 — не короче)
```

Ответ: до Б — 4, до В — 1, до Г — 3.

## Смотрим, как это работает

### Пример 1. Дейкстра целиком

In [ ]:
граф = {
    "А": {"Б": 4, "В": 1},
    "Б": {"А": 4, "Г": 2},
    "В": {"А": 1, "Г": 2},
    "Г": {"Б": 2, "В": 2},
}


def дейкстра(граф, старт):
    метки = {в: float("inf") for в in граф}
    метки[старт] = 0
    посещённые = set()

    while len(посещённые) < len(граф):
        # шаг 2: непосещённая вершина с наименьшей меткой
        текущая = None
        for вершина in граф:
            if вершина not in посещённые:
                if текущая is None or метки[вершина] < метки[текущая]:
                    текущая = вершина
        if метки[текущая] == float("inf"):
            break                                  # остальные недостижимы

        # шаг 3: пробуем улучшить соседей
        for сосед, вес in граф[текущая].items():
            если_через = метки[текущая] + вес
            if если_через < метки[сосед]:
                метки[сосед] = если_через

        посещённые.add(текущая)

    return метки


print(дейкстра(граф, "А"))

Совпало с таблицей, которую мы вели вручную. Обратите внимание
на строку `if метки[текущая] == float("inf")`: она нужна, если граф
распадается на части — до недостижимых вершин расстояние так
и останется бесконечным.

### Пример 2. Не только расстояние, но и сам маршрут

Чтобы восстановить путь, запоминаем, откуда мы в каждую вершину
пришли.

In [ ]:
def дейкстра_с_путём(граф, старт, финиш):
    метки = {в: float("inf") for в in граф}
    метки[старт] = 0
    откуда = {старт: None}
    посещённые = set()

    while len(посещённые) < len(граф):
        текущая = None
        for вершина in граф:
            if вершина not in посещённые:
                if текущая is None or метки[вершина] < метки[текущая]:
                    текущая = вершина
        if текущая is None or метки[текущая] == float("inf"):
            break

        for сосед, вес in граф[текущая].items():
            if метки[текущая] + вес < метки[сосед]:
                метки[сосед] = метки[текущая] + вес
                откуда[сосед] = текущая

        посещённые.add(текущая)

    if метки[финиш] == float("inf"):
        return метки[финиш], []

    путь = []
    вершина = финиш
    while вершина is not None:
        путь.append(вершина)
        вершина = откуда[вершина]
    путь.reverse()
    return метки[финиш], путь


длина, путь = дейкстра_с_путём(граф, "А", "Г")
print(f"Кратчайший путь А → Г: {' → '.join(путь)}, длина {длина}")

Словарь `откуда` — это дерево кратчайших путей: у каждой вершины
ровно один «родитель», через которого до неё выгоднее всего добраться.

### Пример 3. Задача формата ЕГЭ

> В таблице указана протяжённость дорог между пунктами. Найдите
> длину кратчайшего пути из А в Ж.

In [ ]:
пункты = ["А", "Б", "В", "Г", "Д", "Е", "Ж"]
таблица = [
    [0, 4, 7, 0, 0, 0, 0],
    [4, 0, 2, 5, 0, 0, 0],
    [7, 2, 0, 3, 6, 0, 0],
    [0, 5, 3, 0, 2, 8, 0],
    [0, 0, 6, 2, 0, 3, 9],
    [0, 0, 0, 8, 3, 0, 4],
    [0, 0, 0, 0, 9, 4, 0],
]


def из_таблицы(таблица, имена):
    граф = {}
    for i, имя in enumerate(имена):
        граф[имя] = {имена[j]: таблица[i][j]
                     for j in range(len(имена)) if таблица[i][j] > 0}
    return граф


дороги = из_таблицы(таблица, пункты)
метки = дейкстра(дороги, "А")

for пункт in пункты:
    print(f"А → {пункт}: {метки[пункт]}")

длина, путь = дейкстра_с_путём(дороги, "А", "Ж")
print(f"\nМаршрут: {' → '.join(путь)}, длина {длина}")

Ноль в таблице означает «дороги нет» — именно так мы его и прочитали
в функции `из_таблицы`. Если бы в задаче встретилась дорога нулевой
длины, пришлось бы обозначать отсутствие иначе, например `None`.

### Пример 4. Путь через обязательный пункт

> Найдите длину кратчайшего пути из А в Ж, проходящего через В.

Приём: разбить путь на две части. Кратчайший путь через заданную
точку — это кратчайший путь до неё плюс кратчайший путь после.

In [ ]:
до_в = дейкстра(дороги, "А")["В"]
от_в = дейкстра(дороги, "В")["Ж"]

print(f"А → В: {до_в}")
print(f"В → Ж: {от_в}")
print(f"Через В: {до_в + от_в}")
print(f"Без ограничения: {дейкстра(дороги, 'А')['Ж']}")

Дополнительное условие никогда не улучшает ответ — оно либо
не меняет его, либо ухудшает.

### Пример 5. Почему нельзя жадничать

Кажется, что можно проще: на каждом шаге идти по самому короткому
ребру. Проверим эту идею на нашем графе.

In [ ]:
def жадный_путь(граф, старт, финиш):
    текущая = старт
    пройдено = 0
    путь = [старт]
    посещённые = {старт}
    while текущая != финиш:
        варианты = {с: в for с, в in граф[текущая].items() if с not in посещённые}
        if not варианты:
            return None, путь                     # зашли в тупик
        следующая = min(варианты, key=lambda с: варианты[с])
        пройдено += варианты[следующая]
        текущая = следующая
        посещённые.add(текущая)
        путь.append(текущая)
    return пройдено, путь


жадно, маршрут = жадный_путь(дороги, "А", "Ж")
правильно = дейкстра(дороги, "А")["Ж"]

print(f"Жадный ход:  {' → '.join(маршрут)}, длина {жадно}")
print(f"Дейкстра:    {правильно}")

Жадный выбор ближайшего соседа заводит не туда: локально выгодный шаг
не гарантирует выгодного маршрута. Дейкстра тоже выбирает минимум,
но не среди соседей, а среди **всех** непосещённых вершин — в этом
и разница.

## Пробуем сами

Работаем с графом `дороги` из примера 3.

### Задача 1. Кратчайший путь до Ж

Чему равна длина кратчайшего пути из А в Ж?

In [ ]:
#@title 🧩 Задача 1. Длина пути { display-mode: "form" }
#@markdown Впишите число
длина_аж = 0 #@param {type:"integer"}

si.ответ("1", длина_аж, "4ec9599fc203d176",
         hint="Загляните в вывод примера 3.")

### Задача 2. Дейкстра своими руками

Функция получает граф в виде словаря «вершина → {сосед: вес}»
и стартовую вершину, возвращает словарь кратчайших расстояний.
До недостижимых вершин верните `float("inf")`.

In [ ]:
def мой_дейкстра(граф, старт):
    return ...

In [ ]:
si.check("2", мой_дейкстра, [
    (({"А": {"Б": 4, "В": 1}, "Б": {"А": 4, "Г": 2},
       "В": {"А": 1, "Г": 2}, "Г": {"Б": 2, "В": 2}}, "А"),
     {"А": 0, "Б": 4, "В": 1, "Г": 3}),
    (({"А": {"Б": 5}, "Б": {"А": 5}, "В": {}}, "А"),
     {"А": 0, "Б": 5, "В": float("inf")}),
    (({"А": {}}, "А"), {"А": 0}),
])

### Задача 3. Расстояние между двумя пунктами

Функция возвращает длину кратчайшего пути между двумя вершинами.
Если пути нет — `-1`.

In [ ]:
def расстояние(граф, старт, финиш):
    return ...

In [ ]:
si.check("3", расстояние, [
    (({"А": {"Б": 4, "В": 1}, "Б": {"А": 4, "Г": 2},
       "В": {"А": 1, "Г": 2}, "Г": {"Б": 2, "В": 2}}, "А", "Г"), 3),
    (({"А": {"Б": 5}, "Б": {"А": 5}, "В": {}}, "А", "В"), -1),
    (({"А": {"Б": 2}, "Б": {"А": 2}}, "А", "А"), 0),
])

### Задача 4. Путь через обязательный пункт

Функция возвращает длину кратчайшего пути из старта в финиш,
проходящего через заданную вершину.

In [ ]:
def через(граф, старт, промежуточная, финиш):
    return ...

In [ ]:
si.check("4", через, [
    (({"А": {"Б": 4, "В": 1}, "Б": {"А": 4, "Г": 2},
       "В": {"А": 1, "Г": 2}, "Г": {"Б": 2, "В": 2}}, "А", "Б", "Г"), 6),
    (({"А": {"Б": 1}, "Б": {"А": 1, "В": 1}, "В": {"Б": 1}}, "А", "Б", "В"), 2),
])

### Задача 5. Матрица в словарь с весами

Функция получает весовую матрицу и имена вершин, возвращает граф
в виде словаря «вершина → {сосед: вес}». Ноль означает отсутствие
дороги.

In [ ]:
def из_матрицы(матрица, имена):
    return ...

In [ ]:
si.check("5", из_матрицы, [
    (([[0, 3], [3, 0]], ["А", "Б"]), {"А": {"Б": 3}, "Б": {"А": 3}}),
    (([[0, 0], [0, 0]], ["А", "Б"]), {"А": {}, "Б": {}}),
    (([[0, 1, 5], [1, 0, 0], [5, 0, 0]], ["А", "Б", "В"]),
     {"А": {"Б": 1, "В": 5}, "Б": {"А": 1}, "В": {"А": 5}}),
])

### Задача 6. Когда Дейкстра ломается

При каких весах алгоритм Дейкстры может дать неверный ответ?

In [ ]:
#@title 🧩 Задача 6. Ограничение алгоритма { display-mode: "form" }
#@markdown Выберите ответ
ограничение = "выбери ответ" #@param ["выбери ответ", "при отрицательных весах", "при очень больших весах", "при дробных весах"]

si.ответ("6", ограничение, "d5859676c648d1c7",
         hint="Алгоритм опирается на то, что путь может только удлиняться.")

### Задача 7. Самый дальний пункт

Функция возвращает имя вершины, до которой от старта дальше всего
(среди достижимых). Если таких несколько — первую по алфавиту.

In [ ]:
def самая_дальняя(граф, старт):
    return ...

In [ ]:
si.check("7", самая_дальняя, [
    (({"А": {"Б": 4, "В": 1}, "Б": {"А": 4, "Г": 2},
       "В": {"А": 1, "Г": 2}, "Г": {"Б": 2, "В": 2}}, "А"), "Б"),
    (({"А": {"Б": 1}, "Б": {"А": 1}}, "А"), "Б"),
])

## Домашнее задание

### Домашнее задание 1. Стоимость маршрута

Функция получает граф и список вершин-маршрут, возвращает суммарный
вес пути. Если какого-то ребра нет — верните `-1`.

In [ ]:
def стоимость(граф, маршрут):
    return ...

In [ ]:
si.check("дз1", стоимость, [
    (({"А": {"Б": 4, "В": 1}, "Б": {"А": 4, "Г": 2},
       "В": {"А": 1, "Г": 2}, "Г": {"Б": 2, "В": 2}}, ["А", "В", "Г"]), 3),
    (({"А": {"Б": 4}, "Б": {"А": 4}, "В": {}}, ["А", "В"]), -1),
    (({"А": {"Б": 4}, "Б": {"А": 4}}, ["А"]), 0),
])

### Домашнее задание 2. Все расстояния

Функция возвращает словарь «вершина → расстояние до неё от старта»,
в котором недостижимые вершины вообще отсутствуют.

In [ ]:
def только_достижимые(граф, старт):
    return ...

In [ ]:
si.check("дз2", только_достижимые, [
    (({"А": {"Б": 5}, "Б": {"А": 5}, "В": {}}, "А"), {"А": 0, "Б": 5}),
    (({"А": {}}, "А"), {"А": 0}),
])

### Домашнее задание 3. Свой маршрут

Возьмите пять мест в своём городе, между которыми ходите, и запишите
граф с реальным временем в пути в минутах. Найдите программой самый
быстрый маршрут между двумя крайними точками и сравните с тем, что
предлагает карта в телефоне. Если ответы разошлись — объясните почему
(пробки? пересадки? другой набор дорог?).

---

### Любопытно

В навигаторах чистый Дейкстра не используется: перебирать всю карту
страны ради поездки в соседний район слишком дорого. Работают его
ускоренные потомки — алгоритмы, которые заранее просчитывают
«магистральные» связи и во время поиска сразу тянут маршрут
в нужную сторону. Основа при этом та же самая: метка вершины
и попытка её улучшить.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 11](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-11.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 13 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-13.ipynb)